In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.5 MB/s eta 0:00:00a 0:00:01


In [3]:
# ============================================================
# CELL 1 — Install dependencies (run once, then restart runtime)
# ============================================================
import torch
v = torch.__version__.split("+")[0].replace(".", "")[:3]
# cu = "cu121"   # change to cu118 if your Colab gives CUDA 11.8
!pip install torch_geometric -q
# !pip install torch_scatter torch_sparse torch_cluster torch_spline_conv \
#   -f https://data.pyg.org/whl/torch-{torch.__version__}+{cu}.html -q

In [2]:
# ============================================================
# CELL 1 — Imports & Config
# ============================================================
import json, pickle, random, time
import numpy as np
import pandas as pd
 
import torch
import torch.nn as nn
import torch.nn.functional as F
 
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, GCNConv, global_mean_pool, SAGPooling
from torch.utils.data import Dataset, DataLoader as TorchDataLoader
 
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score, precision_score, recall_score
)
 
# ── Hyper-params ─────────────────────────────────────────────
HIDDEN_DIM      = 64    # GAT hidden dim
HEADS           = 2     # GATConv attention heads
EMBED_DIM       = 64    # GCN / predictor dim
GAT_BATCH       = 32    # proteins per GAT mini-batch (lower = less VRAM)
LR              = 1e-3
WEIGHT_DECAY    = 1e-4
EPOCHS          = 80
BATCH_SIZE      = 128   # complex-level batch
GAT_UNFREEZE_EP = 20    # epoch to start fine-tuning GAT (set 999 to never)
SEED            = 42
# ─────────────────────────────────────────────────────────────
 
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU  :", torch.cuda.get_device_name(0))
    print("VRAM :", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

Device: cuda
GPU  : Tesla T4
VRAM : 15.6 GB


In [3]:
# ============================================================
# CELL 2 — Load data
# ============================================================
 
# 2a. Protein index map
with open("/kaggle/input/datasets/shubhamkumar108/protein/protein_index_map.json") as f:
    protein_index_map = json.load(f)
idx_to_pid    = {v: k for k, v in protein_index_map.items()}
N_PROTEINS    = len(protein_index_map)
valid_indices = set(range(N_PROTEINS))
print(f"Proteins in index : {N_PROTEINS}")
 
# 2b. Protein graphs
with open("/kaggle/input/datasets/shubhamkumar108/protein/proteinGraphsIndexed.pkl", "rb") as f:
    protein_graphs = pickle.load(f)
print(f"Protein graphs    : {len(protein_graphs)}")
 
# 2c. PPI edges
ppi_df = pd.read_csv("/kaggle/input/datasets/shubhamkumar108/protein/positiveEdges_indexed.csv")
src, dst = ppi_df["Node1"].tolist(), ppi_df["Node2"].tolist()
ppi_edge_index = torch.tensor([src+dst, dst+src], dtype=torch.long).to(device)
print(f"PPI edges (bi)    : {ppi_edge_index.shape[1]}")
 
# 2d. Positive complexes
with open("/kaggle/input/datasets/shubhamkumar108/protein/indexed_complexes.json") as f:
    pos_complexes_raw = json.load(f)
pos_complexes = [
    cx for cx in pos_complexes_raw
    if len(cx) >= 2 and all(i in valid_indices for i in cx)
]
print(f"Positive complexes: {len(pos_complexes)}")
 
# 2e. Negative complexes — map file, fallback to random sampling
with open("/kaggle/input/datasets/shubhamkumar108/protein/N_RANDOM_Comb.json") as f:
    neg_complexes_raw = json.load(f)
 
neg_complexes = []
for cx in neg_complexes_raw:
    mapped, skip = [], False
    for m in cx:
        key = str(m).strip()
        if key in protein_index_map:
            mapped.append(protein_index_map[key])
        else:
            skip = True; break
    if not skip and len(mapped) >= 2:
        neg_complexes.append(mapped)
 
print(f"Neg from file     : {len(neg_complexes)}")
 
target   = len(pos_complexes)
pos_sets = set(frozenset(cx) for cx in pos_complexes)
all_idx  = list(valid_indices)
rng      = random.Random(SEED)
 
while len(neg_complexes) < target:
    size  = rng.choice([2, 3, 4])
    combo = rng.sample(all_idx, size)
    if frozenset(combo) not in pos_sets:
        neg_complexes.append(combo)
 
if len(neg_complexes) > target:
    neg_complexes = random.sample(neg_complexes, target)
 
print(f"Final neg complexes: {len(neg_complexes)}")

Proteins in index : 7499
Protein graphs    : 7499
PPI edges (bi)    : 37528
Positive complexes: 2940
Neg from file     : 0
Final neg complexes: 2940


In [4]:
# ============================================================
# CELL 3 — Split & DataLoaders
# ============================================================
def split_list(lst, label, train_r=0.70, val_r=0.15, seed=42):
    data = [(cx, label) for cx in lst]
    random.Random(seed).shuffle(data)
    n = len(data); t1 = int(n*train_r); t2 = int(n*(train_r+val_r))
    return data[:t1], data[t1:t2], data[t2:]
 
pos_tr, pos_va, pos_te = split_list(pos_complexes, 1)
neg_tr, neg_va, neg_te = split_list(neg_complexes, 0)
 
train_data = pos_tr + neg_tr; random.shuffle(train_data)
val_data   = pos_va + neg_va; random.shuffle(val_data)
test_data  = pos_te + neg_te
 
print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")
 
class ComplexDataset(Dataset):
    def __init__(self, d): self.d = d
    def __len__(self):     return len(self.d)
    def __getitem__(self, i): return self.d[i]
 
def collate_fn(batch): return batch
 
train_loader = TorchDataLoader(ComplexDataset(train_data), BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = TorchDataLoader(ComplexDataset(val_data),   BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = TorchDataLoader(ComplexDataset(test_data),  BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

Train: 4116  Val: 882  Test: 882


In [5]:
# ============================================================
# CELL 4 — Models
# ============================================================
 
class GAT1(nn.Module):
    """Structure encoder: one protein graph → [1, hidden_dim]."""
 
    def __init__(self, input_dim=24, hidden_dim=HIDDEN_DIM, heads=HEADS):
        super().__init__()
        self.fc1   = nn.Linear(input_dim, hidden_dim)
        self.conv1 = GATConv(hidden_dim, hidden_dim, heads=heads, concat=False, edge_dim=1)
        self.conv2 = GATConv(hidden_dim, hidden_dim, heads=heads, concat=False, edge_dim=1)
        self.conv3 = GATConv(hidden_dim, hidden_dim, heads=heads, concat=False, edge_dim=1)
        self.pool1 = SAGPooling(hidden_dim)
        self.pool2 = SAGPooling(hidden_dim)
        self.pool3 = SAGPooling(hidden_dim)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.bn2   = nn.BatchNorm1d(hidden_dim)
        self.bn3   = nn.BatchNorm1d(hidden_dim)
 
    def forward(self, data):
        x         = data.x.float()
        ei        = data.edge_index
        ea        = data.edge_attr.float() if data.edge_attr is not None else None
        batch     = data.batch
 
        x = self.fc1(x)
 
        x_r = x; x = self.conv1(x, ei, edge_attr=ea); x = self.bn1(x); x = F.relu(x + x_r)
        x, ei, ea, batch, _, _ = self.pool1(x, ei, edge_attr=ea, batch=batch)
 
        x_r = x; x = self.conv2(x, ei, edge_attr=ea); x = self.bn2(x); x = F.relu(x + x_r)
        x, ei, ea, batch, _, _ = self.pool2(x, ei, edge_attr=ea, batch=batch)
 
        x_r = x; x = self.conv3(x, ei, edge_attr=ea); x = self.bn3(x); x = F.relu(x + x_r)
        x, ei, ea, batch, _, _ = self.pool3(x, ei, edge_attr=ea, batch=batch)
 
        return global_mean_pool(x, batch)   # [B, hidden_dim]
 
 
class GCN_Refiner(nn.Module):
    """2-layer GCN that refines protein embeddings with PPI context."""
 
    def __init__(self, in_dim=HIDDEN_DIM, out_dim=EMBED_DIM):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)
        self.gcn1 = GCNConv(out_dim, out_dim)
        self.gcn2 = GCNConv(out_dim, out_dim)
        self.bn1  = nn.BatchNorm1d(out_dim)
        self.bn2  = nn.BatchNorm1d(out_dim)
 
    def forward(self, X, ppi_edge_index):
        X = self.proj(X)
        X = self.bn1(F.relu(self.gcn1(X, ppi_edge_index)))
        X = self.bn2(F.relu(self.gcn2(X, ppi_edge_index)))
        return X                            # [N, EMBED_DIM]
 
 
class ComplexPredictor(nn.Module):
    """Mean-pool member embeddings → MLP → binary logit."""
 
    def __init__(self, dim=EMBED_DIM):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,  64), nn.ReLU(),
            nn.Linear( 64,   1)
        )
 
    def forward(self, H, complex_indices_batch):
        logits = []
        for members in complex_indices_batch:
            idx    = torch.tensor(members, dtype=torch.long, device=H.device)
            pooled = H[idx].mean(0)
            logits.append(self.mlp(pooled))
        return torch.cat(logits, dim=0)

In [6]:
# ============================================================
# CELL 5 — Instantiate
# ============================================================
gat_enc   = GAT1().to(device)
gcn_ref   = GCN_Refiner().to(device)
predictor = ComplexPredictor().to(device)
 
# Separate optimisers so we can freeze/unfreeze GAT independently
opt_gcn_pred = torch.optim.Adam(
    list(gcn_ref.parameters()) + list(predictor.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY)
 
opt_gat = torch.optim.Adam(
    gat_enc.parameters(), lr=LR*0.1, weight_decay=WEIGHT_DECAY)
 
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt_gcn_pred, T_max=EPOCHS)
 
n_params = (sum(p.numel() for p in gat_enc.parameters()) +
            sum(p.numel() for p in gcn_ref.parameters()) +
            sum(p.numel() for p in predictor.parameters()))
print(f"Total parameters: {n_params:,}")

Total parameters: 58,055


In [7]:
# ============================================================
# CELL 6 — Key helper: pre-encode all proteins with GAT
#            Returns a CPU tensor (moves to device per batch)
# ============================================================
@torch.no_grad()
def precompute_gat_embeddings():
    """
    Run GAT1 over all protein graphs in small chunks.
    No gradients → computation graph is never built → zero OOM risk.
    Returns: raw_embs [N, HIDDEN_DIM]  on CPU
    """
    gat_enc.eval()
    chunks = [protein_graphs[i:i+GAT_BATCH]
              for i in range(0, len(protein_graphs), GAT_BATCH)]
    all_emb = []
    for chunk in chunks:
        pyg_batch = Batch.from_data_list(chunk).to(device)
        emb = gat_enc(pyg_batch)            # [chunk_size, HIDDEN_DIM]
        all_emb.append(emb.cpu())           # immediately move to CPU
    return torch.cat(all_emb, dim=0)        # [N, HIDDEN_DIM]  on CPU

In [8]:
# ============================================================
# CELL 7 — Eval helper
# ============================================================
@torch.no_grad()
def evaluate(loader, H_device):
    """H_device: [N, EMBED_DIM] already on device."""
    gcn_ref.eval(); predictor.eval()
    all_logits, all_labels = [], []
    for batch in loader:
        members_list = [cx  for cx, _   in batch]
        labels       = torch.tensor([lbl for _, lbl in batch],
                                    dtype=torch.float, device=device)
        logits = predictor(H_device, members_list)
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())
 
    L = torch.cat(all_logits).numpy()
    Y = torch.cat(all_labels).numpy()
    P = torch.sigmoid(torch.tensor(L)).numpy()
    pred = (P >= 0.5).astype(int)
 
    return {
        "loss"     : F.binary_cross_entropy_with_logits(torch.tensor(L), torch.tensor(Y)).item(),
        "auc"      : roc_auc_score(Y, P),
        "ap"       : average_precision_score(Y, P),
        "acc"      : accuracy_score(Y, pred),
        "f1"       : f1_score(Y, pred, zero_division=0),
        "precision": precision_score(Y, pred, zero_division=0),
        "recall"   : recall_score(Y, pred, zero_division=0),
    }

In [10]:
# ============================================================
# CELL 8 — Training loop  (replace only this cell)
# ============================================================
# Fix: Instead of retain_graph=True (which breaks with BatchNorm inplace ops),
#      recompute H = gcn_ref(raw_embs_gpu) fresh for each mini-batch.
#      GCN over 7499 nodes is microseconds — no memory or speed penalty.
# ============================================================

CKPT     = "/kaggle/working/best_complex_model.pt"
best_auc = 0.0
history  = []

print("Pre-computing GAT embeddings (no_grad)...")
raw_embs_cpu = precompute_gat_embeddings()   # [7499, 64] on CPU
print(f"  done. shape={raw_embs_cpu.shape}")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    gat_fine_tune = (epoch >= GAT_UNFREEZE_EP)

    # ── Move raw embeddings to GPU once per epoch ────────────
    raw_embs_gpu = raw_embs_cpu.to(device)   # [N, HIDDEN_DIM]

    gcn_ref.train(); predictor.train()
    if gat_fine_tune:
        gat_enc.train()
    else:
        gat_enc.eval()

    epoch_loss = 0.0

    for batch in train_loader:
        members_list = [cx  for cx, _   in batch]
        labels       = torch.tensor([lbl for _, lbl in batch],
                                    dtype=torch.float, device=device)

        # ── Recompute H fresh each batch (no retain_graph needed) ──
        # Phase A: raw_embs_gpu has no grad (came from no_grad pre-encode)
        #          so GCN graph is small and self-contained each step
        H      = gcn_ref(raw_embs_gpu, ppi_edge_index)   # [N, EMBED_DIM]
        logits = predictor(H, members_list)
        loss   = F.binary_cross_entropy_with_logits(logits, labels)

        opt_gcn_pred.zero_grad()
        if gat_fine_tune:
            opt_gat.zero_grad()

        loss.backward()   # ← no retain_graph, no inplace conflict

        torch.nn.utils.clip_grad_norm_(
            list(gcn_ref.parameters()) + list(predictor.parameters()), 1.0)
        if gat_fine_tune:
            torch.nn.utils.clip_grad_norm_(gat_enc.parameters(), 1.0)

        opt_gcn_pred.step()
        if gat_fine_tune:
            opt_gat.step()

        epoch_loss += loss.item()

    scheduler.step()

    # ── Validation ───────────────────────────────────────────
    with torch.no_grad():
        gcn_ref.eval()
        H_val = gcn_ref(raw_embs_gpu, ppi_edge_index).detach()

    val_m    = evaluate(val_loader, H_val)
    avg_loss = epoch_loss / max(len(train_loader), 1)
    history.append({"epoch": epoch, "train_loss": avg_loss,
                    **{f"val_{k}": v for k, v in val_m.items()}})

    # ── Refresh GAT embeddings once per epoch in Phase B ─────
    if gat_fine_tune:
        raw_embs_cpu = precompute_gat_embeddings()

    if epoch % 5 == 0 or epoch == 1:
        phase = "GAT+GCN" if gat_fine_tune else "GCN only"
        print(f"Ep {epoch:3d} [{phase}] | loss={avg_loss:.4f} | "
              f"auc={val_m['auc']:.4f} | f1={val_m['f1']:.4f} | "
              f"acc={val_m['acc']:.4f} | {time.time()-t0:.1f}s")

    if val_m["auc"] > best_auc:
        best_auc = val_m["auc"]
        torch.save({"gat_enc"  : gat_enc.state_dict(),
                    "gcn_ref"  : gcn_ref.state_dict(),
                    "predictor": predictor.state_dict(),
                    "epoch"    : epoch,
                    "val_auc"  : best_auc}, CKPT)

    if device.type == "cuda":
        torch.cuda.empty_cache()

print(f"\nBest val AUC: {best_auc:.4f}  →  {CKPT}")

Pre-computing GAT embeddings (no_grad)...
  done. shape=torch.Size([7499, 64])
Ep   1 [GCN only] | loss=0.5616 | auc=0.6583 | f1=0.1032 | acc=0.5272 | 4.6s
Ep   5 [GCN only] | loss=0.3952 | auc=0.7239 | f1=0.1660 | acc=0.5442 | 4.6s
Ep  10 [GCN only] | loss=0.3566 | auc=0.8500 | f1=0.7722 | acc=0.7472 | 4.6s
Ep  15 [GCN only] | loss=0.3496 | auc=0.8634 | f1=0.7308 | acc=0.7619 | 4.6s
Ep  20 [GAT+GCN] | loss=0.3072 | auc=0.8097 | f1=0.7350 | acc=0.6837 | 8.3s
Ep  25 [GAT+GCN] | loss=0.3013 | auc=0.8216 | f1=0.7094 | acc=0.7324 | 8.1s
Ep  30 [GAT+GCN] | loss=0.2724 | auc=0.8454 | f1=0.2564 | acc=0.5726 | 8.2s
Ep  35 [GAT+GCN] | loss=0.2541 | auc=0.8841 | f1=0.4704 | acc=0.6451 | 8.1s
Ep  40 [GAT+GCN] | loss=0.2457 | auc=0.9081 | f1=0.7822 | acc=0.8061 | 8.1s
Ep  45 [GAT+GCN] | loss=0.2292 | auc=0.8466 | f1=0.4585 | acc=0.6304 | 8.2s
Ep  50 [GAT+GCN] | loss=0.2160 | auc=0.9149 | f1=0.7951 | acc=0.8107 | 8.2s
Ep  55 [GAT+GCN] | loss=0.2051 | auc=0.9248 | f1=0.8034 | acc=0.8186 | 8.2s
Ep  6

In [12]:
# ============================================================
# CELL 9 — Test evaluation
# ============================================================
ckpt = torch.load(CKPT, map_location=device, weights_only=False)
gat_enc.load_state_dict(ckpt["gat_enc"])
gcn_ref.load_state_dict(ckpt["gcn_ref"])
predictor.load_state_dict(ckpt["predictor"])
print(f"Loaded checkpoint  epoch={ckpt['epoch']}  val_auc={ckpt['val_auc']:.4f}")
 
with torch.no_grad():
    raw  = precompute_gat_embeddings()
    gcn_ref.eval()
    H_test = gcn_ref(raw.to(device), ppi_edge_index).detach()
 
test_m = evaluate(test_loader, H_test)
print("\n===== TEST RESULTS =====")
for k, v in test_m.items():
    print(f"  {k:12s}: {v:.4f}")

Loaded checkpoint  epoch=70  val_auc=0.9526

===== TEST RESULTS =====
  loss        : 0.2871
  auc         : 0.9521
  ap          : 0.9496
  acc         : 0.8878
  f1          : 0.8881
  precision   : 0.8851
  recall      : 0.8912


In [ ]:
# ============================================================
# CELL 10 — Inference helper
# ============================================================
def predict_complex(member_indices, threshold=0.5):
    gat_enc.eval(); gcn_ref.eval(); predictor.eval()
    with torch.no_grad():
        raw   = precompute_gat_embeddings()
        H     = gcn_ref(raw.to(device), ppi_edge_index)
        logit = predictor(H, [member_indices])
        prob  = torch.sigmoid(logit).item()
    label = "COMPLEX" if prob >= threshold else "NOT COMPLEX"
    names = [idx_to_pid.get(i, str(i)) for i in member_indices]
    print(f"{names}  →  prob={prob:.4f}  →  {label}")
    return prob
 
# Example: predict_complex([7442, 2193, 2580])

In [ ]:
# ============================================================
# CELL 11 — Plot
# ============================================================
import matplotlib.pyplot as plt
df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(df.epoch, df.train_loss, label="train"); axes[0].plot(df.epoch, df.val_loss, label="val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(df.epoch, df.val_auc, label="AUC"); axes[1].plot(df.epoch, df.val_f1, label="F1")
axes[1].set_title("Val Metrics"); axes[1].legend()
plt.tight_layout(); plt.savefig("/kaggle/working/training_curves.png", dpi=150); plt.show()